In [1]:
import os

In [2]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.messages import ToolMessage
from langchain_core.rate_limiters import InMemoryRateLimiter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Dict, Optional
import gradio as gr
import uuid

C:\Users\User\AppData\Local\Temp\ipykernel_29640\1806102346.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import GoogleSerperRun


In [3]:
from langchain.agents.middleware import AgentMiddleware, ModelCallLimitMiddleware, PIIMiddleware, TodoListMiddleware

In [4]:
class TolerateToolErrors(AgentMiddleware):
    """Hand tool failures back to the model as a message so it can recover, rather than
    crashing the run. Tools that touch the outside world, like a browser, fail now and then."""

    async def awrap_tool_call(self, request, handler):
        try:
            return await handler(request)
        except Exception as error:
            return ToolMessage(
                content=f"That tool call failed: {error}. Try another approach.",
                tool_call_id=request.tool_call["id"],
            )

In [5]:
client = MultiServerMCPClient(
    {
        "playwright": {
            "transport": "stdio",
            "command": "npx",
            "args": ["@playwright/mcp@latest", "--isolated"]
        }
    }
)

Helps with window opening issue, doesnt need to be in actual app.

In [6]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


In [7]:
load_dotenv(override=True)

True

In [8]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

In [9]:
class SupermarketEssentials(BaseModel):
    smarket: str = Field(description="The Supermarkets name")
    prices: dict[str, Optional[str]] = Field(description="Item name to price. Keys must be exactly: milk_2L, cheese_1kg, mince_500g, eggs_6_pack. Use null if an item couldn't be found.")
    total_price: Optional[str] = Field(description="The total cost of the essentials at this supermarket, or null if incomplete")

class Essentials(BaseModel):
    region: str = Field(description="The region in which the prices are pulled from")
    supermarkets: Dict[str, SupermarketEssentials] = Field(description="Keyed by supermarket name: woolworthes, paknsave, new_world")
    cheapest: str = Field(description="The name of the cheapest total supermarket")


In [10]:
mcp_tools = await client.get_tools()

ALLOWED_TOOLS = {
    "browser_navigate",
    "browser_snapshot",   # or browser_get_text / accessibility tree tool
    "browser_click",
    "browser_wait_for",
    "browser_take_screenshot",  # optional, for debugging
}

mcp_tools = [t for t in mcp_tools if t.name in ALLOWED_TOOLS]
tools = [search] + mcp_tools

In [11]:
SYSTEM_PROMPT = f"""You are a helpful supermarket assistant tasked with finding prices from the web.
Use the region given to you to find the prices or default to any Auckland store and run continue do not stop. Use non member prices.

You explore different supermarket websites in New Zealand (Woolworthes, PAK'nSAVE, New World) finding the prices for these essential items:
standard blue top milk(2 litres), cheese(1kg), beef mince(500g) 5% fat, eggs(6 pack). 

Get the cheapest branded item but make sure it conforms to the specified weight/amount shown in brackets.

Below that provide a message warning about price variability and innaccuracies if you think there might be any."""

In [12]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.5,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [13]:
memory = InMemorySaver()

In [14]:
model = ChatOpenAI(
    model="gpt-5.4-mini",
    rate_limiter=rate_limiter
)

In [15]:
price_agent = create_agent(
    model=model, 
    tools=tools, 
    system_prompt=SYSTEM_PROMPT, 
    middleware=[TolerateToolErrors(), TodoListMiddleware(), PIIMiddleware("email"), PIIMiddleware("credit_card", apply_to_tool_results=True), ModelCallLimitMiddleware(run_limit=30)],
    checkpointer=memory,
    response_format=Essentials
    )

In [16]:
def format_essentials(essentials: Essentials) -> str:
    lines = [f"## Essentials comparison - {essentials.region}\n"]
    for name, sm in essentials.supermarkets.items():
        lines.append(f"### {sm.smarket}")
        for item, price in sm.prices.items():
            lines.append(f"- **{item}**: {price if price else 'N/A'}\n")
        lines.append(f"**Total: {sm.total_price if sm.total_price else 'N/A'}**\n")
    lines.append(f"---\n**Cheapest overall: {essentials.cheapest}**")
    return "\n".join(lines)

In [17]:
async def get_prices(region: str):
    config = {"configurable": {"thread_id": f"big3-{uuid.uuid4()}"}}
    result = await price_agent.ainvoke({"messages": [{"role": "user", "content": f"Get the essentials from these New Zealand Supermarkets: 'Woolworthes', 'PAK'nSave', and 'New World' in this region: {region}"}]}, config)
    essentials: Essentials = result["structured_response"]
    return format_essentials(essentials)

In [40]:
with open("custom.css") as f:
    custom_css = f.read()

In [29]:
with gr.Blocks() as demo:
    gr.HTML("""
    <div class="app-header">
        <div class="brand-mark" aria-label="Brand mark">
            <span class="brand-dot brand-green"></span>
            <span class="brand-dot brand-yellow"></span>
            <span class="brand-dot brand-red"></span>
        </div>
        <div class="app-title-text">Supermarket Selector</div>
    </div>
    """, elem_id="app-title")
    with gr.Row():
        results = gr.Markdown("results...", height=325, elem_id="results-box")
    with gr.Column():
        with gr.Group(elem_id="region-wrapper"):
            region = gr.Dropdown(
                value="Auckland",
                label="Region:",
                choices=["Auckland", "Wellington", "Christchurch", "Dunedin"],
                elem_id="region-dropdown",
            )
            submit_button = gr.Button("Submit", elem_id="submit-btn")

    submit_button.click(get_prices, [region], [results])

In [37]:
demo.close()

Closing server running on port: 7861


In [41]:
demo.launch(css=custom_css, inbrowser=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.
